<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 11 · E11 (auto vs all-table onboard) + E12 (enrichment passes)

**E11** compares the two onboarding pipelines on the same fixture:
- **all-table** — deterministic `/onboard` (models *every* table in scope) → `/enrich`.
- **auto-onboard** — the Copilot reads the glossary and *selects* the tables to model
  (`auto_onboard`), onboarding + enriching in one turn (the product's real flow).

The headline metric is **table-selection precision** (does auto-onboard exclude the
distractors the deterministic path always includes?) alongside graded score, coverage,
and query-time distractor leakage.

**E12** then runs additional Copilot enrichment passes on the auto-onboarded project
(K = 0, 1, 2) to test whether pass 2 helps — the E6 "enrich once" question, re-asked on
the Copilot path.

> Requires Postgres + `WREN_MEMORY_STORE=none` + an agent image new enough to expose
> `mdl-files/bulk-status` (needed to activate Copilot overlay changesets). Each Copilot
> turn makes many LLM calls — this notebook is slow. Bump `TRIALS` for confidence
> (Copilot is non-deterministic; ≥3 recommended).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import eval_v2 as v2
import seagate_scoring as scoring

FIXTURE = 'seagate_multi'
fdir = v2.fixture_dir(FIXTURE)
manifest = v2.load_table_manifest(FIXTURE)

config = ec.EvalConfig.from_env(
    schema_name='seagate_ops',
    results_dir=fdir.parent.parent / 'evaluation' / 'results' / 'seagate_multi',
)
client = v2.AgentClientV2(config, schema_names=['seagate_core', 'seagate_ops'])
me = client.login()
print('Authenticated as:', me.get('username') or me)
pre = client.assert_eval_preconditions(require_postgres=True)
print('DB backend:', pre['backend'])
for w in pre['warnings']:
    print('WARNING:', w)
questions = ec.parse_test_queries(fdir / 'test_queries.md')
glossary = (fdir / 'bi_glossary.md').read_text(encoding='utf-8')
print('Loaded', len(questions), 'graded questions')

In [ ]:
def score_sweep(results):
    verdicts = {r['id']: scoring.score_result(
        r['id'], r['result_rows'], r['answer_summary']) for r in results}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return correct, verdicts

def sql_leaks(results):
    D = manifest['distractor_tables']
    return [(r['id'], v2.sql_references_tables(r['sql'], D))
            for r in results if v2.sql_references_tables(r['sql'], D)]

def profile(label, pid, sweep_name, *, coverage=None):
    res = ec.run_experiment(client, sweep_name, questions, save=False)
    correct, _ = score_sweep(res)
    sel = client.selection_metrics(pid, manifest)
    row = {'label': label, 'graded_of_18': correct,
           'precision': sel['precision'], 'recall': sel['recall'],
           'distractor_inclusions': len(sel['distractor_inclusions']),
           'n_models': len(client.active_model_names(pid)),
           'sql_leaks': len(sql_leaks(res)),
           'coverage': coverage}
    print(label, '->', row)
    return row

## E11 — Pipeline A: deterministic all-table onboard → enrich

In [ ]:
print('archived', client.clean_baseline(), 'project(s)')
projA = client.resolve_project(); pidA = projA['id']
client.onboard(pidA)
rows = []
rows.append(profile('all-table: wren_base', pidA, 'wren_base'))
# deterministic enrichment
doc = client.create_document_from_text(pidA, glossary, 'bi_glossary.md')
client.enrich_round(pidA, doc['id'], wait_coverage=False)
covA = client.wait_for_coverage(pidA)
rows.append(profile('all-table: wren_bi', pidA, 'wren_bi', coverage=covA))

## E11 — Pipeline B: Copilot auto-onboard (selective onboard + enrich)

In [ ]:
print('archived', client.clean_baseline(), 'project(s)')
projB = client.resolve_project(); pidB = projB['id']
auto = client.auto_onboard(pidB, glossary, max_steps=20)
print('auto-onboard: changeset items=%s applied=%s activated=%s err=%s'
      % (auto['items'], auto['applied'], auto['activated'], auto['activate_error']))
covB0 = client.wait_for_coverage(pidB)
rows.append(profile('auto-onboard (K=0)', pidB, 'wren_bi', coverage=covB0))

### E11 verdict — auto vs all-table

In [ ]:
import pandas as pd
e11 = pd.DataFrame(rows)
print(e11.to_string(index=False))
print('\nHypothesis check: auto-onboard precision should exceed the deterministic '
      '~0.583 (it should drop in-schema distractors). Recall < 1 means it MISSED a '
      'relevant table — inspect selection_metrics() if so.')

## E12 — additional Copilot enrichment passes on the auto-onboarded project

K=0 is the auto-onboard state captured above (`covB0`). Now run 1 then 2 extra
refinement passes and watch coverage + graded score.

In [ ]:
TRIALS = 1  # bump for confidence; each pass is a full Copilot turn
e12 = [{'K': 0, 'coverage': covB0,
        'graded_of_18': rows[-1]['graded_of_18'],
        'n_models': rows[-1]['n_models']}]
for k in (1, 2):
    res = client.copilot_enrich_pass(pidB, glossary, max_steps=16)
    sweep = ec.run_experiment(client, 'wren_bi', questions, save=False)
    correct, _ = score_sweep(sweep)
    e12.append({'K': k, 'coverage': res['coverage'],
                'graded_of_18': correct, 'n_models': len(res['active_models']),
                'activated': res['activated'], 'err': res['activate_error']})
    print(f"K={k}: coverage={res['coverage']} graded={correct}/18 "
          f"models={len(res['active_models'])} activated={res['activated']}")

In [ ]:
e12df = pd.DataFrame(e12)
e12df['d_coverage'] = e12df['coverage'].diff()
e12df['d_graded'] = e12df['graded_of_18'].diff()
print(e12df.to_string(index=False))
print('\nE6 predicted plateau after one pass. If d_coverage and d_graded for K=2 are '
      '~0 (or negative), the Copilot path confirms "enrich once".')

**Reading E11/E12.**
- **E11:** the story is *selection*. If auto-onboard's precision is high (distractors
  excluded) with recall 1.0 and a comparable graded score on a smaller MDL, the Copilot
  pipeline is the better default. If recall drops (missed a relevant table) or graded
  falls, selectivity is over-aggressive — report which table it missed.
- **E12:** if K=2 adds nothing over K=1, the E6 "enrich once" finding generalises to the
  Copilot path → the product should not auto-loop enrichment. Average ≥3 trials before
  trusting any single down-tick (the live run showed ±4/18 single-trial variance).